# GSE44076 Sample Metadata

This notebook organizes the sample-level metadata stored in the local GSE44076 series matrix. Its scope is limited to checking metadata fields, defining transparent group labels, and creating a local metadata table for later work.

No expression analysis or biological inference is performed here.

## Setup

The raw series matrix and generated metadata CSV remain local because both `data/raw/` and `data/processed/` are excluded from Git.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.geo_loader import (  # noqa: E402
    build_sample_metadata_table,
    read_geo_series_lines,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "GSE44076_series_matrix.txt.gz"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "sample_metadata.csv"

## Load Sample Metadata

The loader reads the gzipped text safely and aligns repeated GEO sample fields by accession.

In [2]:
if not RAW_PATH.is_file():
    raise FileNotFoundError(
        "GSE44076 series matrix not found. Place the downloaded file at "
        f"{RAW_PATH}"
    )

series_lines = read_geo_series_lines(RAW_PATH)
sample_metadata = build_sample_metadata_table(series_lines)

print(f"Raw file: {RAW_PATH.relative_to(PROJECT_ROOT)}")
print(f"Sample metadata shape: {sample_metadata.shape}")
display(sample_metadata.head())

Raw file: data\raw\GSE44076_series_matrix.txt.gz
Sample metadata shape: (246, 9)


,sample_accession,title,source_name,sample_type,individual_id,stage,location,gender,age
0,GSM1077598,Mucosa sample from A2119 healthy donnor,Healthy colon mucosa cells,Mucosa,A2119,--,Left,Male,62
1,GSM1077599,Mucosa sample from A2142 healthy donnor,Healthy colon mucosa cells,Mucosa,A2142,--,Left,Female,77
2,GSM1077600,Mucosa sample from B2104 healthy donnor,Healthy colon mucosa cells,Mucosa,B2104,--,Left,Female,78
3,GSM1077601,Mucosa sample from B2127 healthy donnor,Healthy colon mucosa cells,Mucosa,B2127,--,Right,Male,65
4,GSM1077602,Mucosa sample from B2150 healthy donnor,Healthy colon mucosa cells,Mucosa,B2150,--,Right,Female,52


## Available Fields

The characteristic columns below come directly from repeated `Sample_characteristics_ch1` records. Placeholder values such as `--` are retained rather than reinterpreted at this stage.

In [3]:
field_summary = pd.DataFrame(
    {
        "non_null": sample_metadata.notna().sum(),
        "unique_values": sample_metadata.nunique(dropna=True),
    }
)
display(field_summary)
display(sample_metadata[["sample_accession", "title", "source_name"]].head(10))

,non_null,unique_values
sample_accession,246,246
title,246,246
source_name,246,3
sample_type,246,3
individual_id,246,148
stage,246,3
location,246,2
gender,246,2
age,246,45


,sample_accession,title,source_name
0,GSM1077598,Mucosa sample from A2119 healthy donnor,Healthy colon mucosa cells
1,GSM1077599,Mucosa sample from A2142 healthy donnor,Healthy colon mucosa cells
2,GSM1077600,Mucosa sample from B2104 healthy donnor,Healthy colon mucosa cells
3,GSM1077601,Mucosa sample from B2127 healthy donnor,Healthy colon mucosa cells
4,GSM1077602,Mucosa sample from B2150 healthy donnor,Healthy colon mucosa cells
5,GSM1077603,Mucosa sample from C2113 healthy donnor,Healthy colon mucosa cells
6,GSM1077604,Mucosa sample from C2136 healthy donnor,Healthy colon mucosa cells
7,GSM1077605,Mucosa sample from D2102 healthy donnor,Healthy colon mucosa cells
8,GSM1077606,Mucosa sample from D2125 healthy donnor,Healthy colon mucosa cells
9,GSM1077607,Mucosa sample from D2148 healthy donnor,Healthy colon mucosa cells


## Inspect and Define Groups

The GEO source names distinguish healthy mucosa, paired normal mucosa, and primary tumor samples. The group labels below are direct, readable aliases for those source values; they are not inferred from expression measurements.

In [4]:
display(sample_metadata["source_name"].value_counts().rename("sample_count").to_frame())
display(pd.crosstab(sample_metadata["source_name"], sample_metadata["sample_type"]))

group_map = {
    "Healthy colon mucosa cells": "healthy_mucosa",
    "Normal distant colon mucosa cells": "paired_normal_mucosa",
    "Primary colon adenocarcinoma cells": "tumor",
}
sample_metadata["group"] = sample_metadata["source_name"].map(group_map)

if sample_metadata["group"].isna().any():
    unmapped = sample_metadata.loc[sample_metadata["group"].isna(), "source_name"].unique()
    raise ValueError(f"Unmapped source names found: {unmapped.tolist()}")

group_counts = sample_metadata["group"].value_counts().rename("sample_count")
display(group_counts.to_frame())

,sample_count
source_name,
Normal distant colon mucosa cells,98
Primary colon adenocarcinoma cells,98
Healthy colon mucosa cells,50


sample_type,Mucosa,Normal,Tumor
source_name,,,
Healthy colon mucosa cells,50,0,0
Normal distant colon mucosa cells,0,98,0
Primary colon adenocarcinoma cells,0,0,98


,sample_count
group,
paired_normal_mucosa,98
tumor,98
healthy_mucosa,50


## Save the Local Metadata Table

The processed CSV is a reproducible derivative of the GEO metadata and remains ignored by Git.

In [5]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sample_metadata.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {sample_metadata.shape[0]} samples and {sample_metadata.shape[1]} columns")
print(f"Output: {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")

Saved 246 samples and 10 columns
Output: data\processed\sample_metadata.csv


## Scope Note

This table records the metadata available in the series matrix and a documented source-based group label. Later work should check these labels against the GEO study design before using them in any statistical analysis.